# ACTIVIDAD 5 | Notebook 01 — Obtención y auditoría preliminar UC3M-LP
**Caso de uso:** control de acceso vehicular mediante visión computacional.  
**Objetivo de este notebook:** comprobar entorno, disponibilidad de recursos, procedencia, integridad del archivo y estructura de las anotaciones **antes** de entrenar.  
**Estado:** pendiente de ejecución. Ninguna métrica presentada aquí es un resultado experimental hasta ejecutar las celdas.

**IA responsable — control transversal:** usar imágenes solo para la finalidad académica y conforme a la licencia; no exponer matrículas identificables en capturas o GitHub; no considerar el resultado apto para abrir barreras; registrar límites, errores, procedencia y decisiones humanas. Indicador IER-IA: la verificación de controles se documenta con evidencias, no con una puntuación asumida.

## CELDA 1 — Configurar entorno y reproducibilidad
*Qué hace:* fija semilla, crea carpetas y reporta almacenamiento disponible y versiones. **La GPU no es necesaria para la auditoría**.

In [ ]:
import os, sys, json, random, platform, shutil, subprocess
from pathlib import Path
import numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)
RAIZ = Path('/content/actividad5') if Path('/content').exists() else Path.cwd() / 'actividad5'
RAW = RAIZ / 'datos' / 'datos_ini'; EVID = RAIZ / 'evidencias'
RAW.mkdir(parents=True, exist_ok=True); EVID.mkdir(parents=True, exist_ok=True)
total, usado, libre = shutil.disk_usage(RAIZ)
print('Python:',sys.version.split()[0],'| Sistema:',platform.platform())
print('Directorio:',RAIZ)
print(f'Espacio libre: {libre/1024**3:.2f} GiB')
print('Semilla:',SEED)
print('GPU (solo informativo):')
subprocess.run(['nvidia-smi'], check=False) if shutil.which('nvidia-smi') else print('No hay GPU disponible; no es necesaria en esta fase.')

## CELDA 2 — Consultar el registro original de Zenodo
*Qué hace:* solicita los metadatos en la API pública, identifica archivos y enlaces. El ID se toma del registro Zenodo utilizado en el proyecto. Si la API no responde, se detiene antes de descargar.

In [ ]:
import requests
ZENODO_RECORD_ID = '17152029'
api = f'https://zenodo.org/api/records/{ZENODO_RECORD_ID}'
r = requests.get(api, timeout=45)
r.raise_for_status()
registro = r.json()
archivos = registro.get('files', [])
print('ID:', registro.get('id'), '| Título:', registro.get('metadata', {}).get('title'))
for f in archivos:
    print('Archivo:', f.get('key'), '| Tamaño (GB):', round(f.get('size',0)/1e9,2), '| checksum:', f.get('checksum'))
    print('URL:', f.get('links',{}).get('self'))
assert archivos, 'La respuesta no incluye archivos. Confirmar el registro oficial antes de continuar.'

## CELDA 3 — Seleccionar archivo y comprobar capacidad (NO descarga todavía)
*Qué hace:* calcula espacio necesario. La copia comprimida puede rondar 4.5 GB; una extracción completa necesita espacio adicional. **No se extraerá todo el dataset en esta fase**.

In [ ]:
import hashlib
candidatos = [f for f in archivos if str(f.get('key','')).lower().endswith('.zip')]
assert len(candidatos)==1, f'Se esperaba 1 ZIP; disponibles: {[f.get("key") for f in archivos]}'
f = candidatos[0]
nombre = Path(f['key']).name
URL = f.get('links',{}).get('self')
assert URL and URL.startswith('https://'), 'Enlace de descarga no válido.'
tamano = int(f.get('size',0))
total, usado, libre = shutil.disk_usage(RAW)
# Margen de 1 GiB para uso temporal y reportes; descargar ZIP sin extraer todo.
minimo = tamano + 1024**3
print('Archivo:', nombre, '| Tamaño:', round(tamano/1024**3,2),'GiB')
print('Espacio libre:',round(libre/1024**3,2),'GiB | Mínimo recomendado:',round(minimo/1024**3,2),'GiB')
if libre < minimo:
    print('DETENER: espacio insuficiente. Libera almacenamiento o utiliza Drive con suficiente capacidad.')
else:
    print('Capacidad preliminar suficiente para almacenar ZIP, sin extracción completa.')

## CELDA 4 — Descarga controlada y comprobación de integridad
**Importante:** esta celda descarga varios GB desde Zenodo al disco de Colab. Configura `DESCARGAR = False` cuando confirmes espacio suficiente. La descarga puede tardar y el entorno temporal puede desconectarse. Si ya existe un ZIP, se verifica su tamaño. Se comprueba MD5 o SHA256 cuando Zenodo lo ofrece.

In [ ]:
from tqdm.auto import tqdm
DESCARGAR = False  # Cambia a True únicamente cuando decidas iniciar la descarga
zip_path = RAW / nombre
if DESCARGAR:
    if shutil.disk_usage(RAW).free < minimo and not zip_path.exists():
        raise RuntimeError('Sin espacio libre suficiente para la descarga.')
    if not zip_path.exists() or zip_path.stat().st_size != tamano:
        temporal = RAW / (nombre + '.part')
        if temporal.exists():
            temporal.unlink()  # Reinicio limpio para evitar archivos parcialmente mezclados
        with requests.get(URL, stream=True, timeout=(30,180), allow_redirects=True) as resp:
            resp.raise_for_status()
            with open(temporal,'wb') as out, tqdm(total=tamano, unit='B', unit_scale=True, desc='Descargando UC3M-LP') as barra:
                for chunk in resp.iter_content(chunk_size=2*1024*1024):
                    if chunk:
                        out.write(chunk); barra.update(len(chunk))
        assert temporal.stat().st_size == tamano, 'Descarga incompleta: revisar conexión.'
        temporal.replace(zip_path)
    checksum = f.get('checksum','') or ''
    if ':' in checksum:
        algo, esperado = checksum.split(':',1)
        if algo in ('md5','sha256'):
            h=hashlib.new(algo)
            with zip_path.open('rb') as entrada:
                for bloque in iter(lambda: entrada.read(8*1024*1024), b''):
                    h.update(bloque)
            assert h.hexdigest().lower()==esperado.lower(), f'El {algo} no coincide; no usar este archivo.'
            print('Integridad verificada mediante',algo)
        else: print('Checksum no verificado, algoritmo no soportado:',algo)
    else: print('El registro no incluye checksum comprobable en este formato.')
    print('ZIP disponible:', zip_path, '| bytes:', zip_path.stat().st_size)
else:
    print('Descarga desactivada. Tras verificar el espacio, modifica DESCARGAR = False y vuelve a ejecutar esta celda.')

## CELDA 5 — Inspección del ZIP sin extraer 4.5 GB
*Qué hace:* lista estructura y cuenta imágenes/anotaciones. Marca rutas peligrosas y entradas defectuosas; no publica imágenes ni matrículas.

In [ ]:
from zipfile import ZipFile, BadZipFile
from collections import Counter
assert zip_path.is_file(), 'Primero completa la CELDA 4.'
with ZipFile(zip_path) as z:
    items = z.infolist()
    extensiones = Counter(Path(x.filename).suffix.lower() for x in items if not x.is_dir())
    peligrosas = [x.filename for x in items if Path(x.filename).is_absolute() or '..' in Path(x.filename).parts]
    print('Entradas:',len(items),'| Tipos:',dict(extensiones))
    print('Primeras 20 rutas:',*[x.filename for x in items[:20]], sep='\n  ')
    print('Rutas inseguras:',len(peligrosas))
    assert not peligrosas, 'ZIP con rutas inseguras: no extraer.'
    print('Verificación de integridad ZIP: opcional, potencialmente lenta; se realizará sobre entradas usadas en auditoría.')

## CELDA 6 — Auditoría preliminar de anotaciones JSON (sin extraer imágenes)
*Qué hace:* inspecciona archivos JSON del ZIP, si están presentes; evalúa estructura, etiquetas y coordenadas contra dimensiones declaradas. Los controles geométricos no reemplazan la verificación visual humana.

In [ ]:
import re
conteo=Counter(); clases=Counter(); hallazgos=Counter(); ejemplos=[]
with ZipFile(zip_path) as z:
    nombres={p.filename for p in z.infolist()}
    jsons=[x for x in z.infolist() if x.filename.lower().endswith('.json') and not x.is_dir()]
    print('JSON encontrados:',len(jsons))
    for entrada in tqdm(jsons,desc='Revisando metadatos'):
        conteo['json']+=1
        try:
            dato=json.loads(z.read(entrada).decode('utf-8-sig'))
            w,h=int(dato['imageWidth']),int(dato['imageHeight'])
            img_path=str(dato.get('imagePath',''))
            # Buscar nombre de imagen en el mismo directorio, sin asumir que stem(JSON) = stem(JPG).
            base=Path(entrada.filename).parent
            esperado=str(base / img_path).replace('\\','/')
            if esperado not in nombres:
                hallazgos['imagePath_sin_coincidencia_relativa']+=1
                if len(ejemplos)<8: ejemplos.append({'json':entrada.filename,'imagePath':img_path})
            for lp in dato.get('lps',[]):
                conteo['matriculas_anotadas']+=1
                for c in lp.get('characters',[]):
                    conteo['caracteres_anotados']+=1
                    clase=str(c.get('char_id',''))
                    bbox=c.get('bbox_coord',[])
                    if not re.fullmatch(r'[A-Z0-9]',clase): hallazgos['etiqueta_fuera_alfanumerico']+=1;continue
                    clases[clase]+=1
                    try:
                        (x1,y1),(x2,y2)=bbox
                        if not (0<=x1<x2<=w and 0<=y1<y2<=h): hallazgos['bbox_fuera_de_rango']+=1
                    except (TypeError,ValueError): hallazgos['bbox_invalido']+=1
        except Exception as ex:
            hallazgos['json_no_interpretable']+=1
            if len(ejemplos)<8: ejemplos.append({'json':entrada.filename,'error':str(ex)[:120]})
print('Conteos:',dict(conteo)); print('Hallazgos:',dict(hallazgos))
print('Clases observadas:',dict(sorted(clases.items())))
print('Ejemplos de anomalías (sin imágenes):',ejemplos)
if not jsons:
    print('No hay JSON dentro del ZIP: verificar estructura y formato real antes de seguir.')

## CELDA 7 — Guardar evidencias e indicador de IA responsable (sin datos personales)
El reporte se genera **solo a partir de la ejecución**. El IER-IA registra controles verificados y pendientes, no asume cumplimiento. No subir matrículas identificables a GitHub ni imágenes sin licencia comprobada.

In [ ]:
from datetime import datetime,timezone
controles = {
 'procedencia_documentada': True,
 'integridad_zip_verificada': bool(DESCARGAR and zip_path.exists() and ':' in str(f.get('checksum',''))),
 'licencia_revisada_para_redistribucion': False,
 'anonimizacion_y_minimizacion_verificadas': False,
 'revision_de_sesgos_y_representatividad': False,
 'supervision_humana_para_control_de_acceso': True, # Regla de diseño: ninguna apertura automática en esta práctica.
}
reporte={
 'fecha_utc':datetime.now(timezone.utc).isoformat(),
 'dataset':'UC3M-LP', 'zenodo_record_id':ZENODO_RECORD_ID,
 'archivo':nombre, 'bytes':zip_path.stat().st_size,
 'resumen':dict(conteo),'clases':dict(sorted(clases.items())),
 'hallazgos':dict(hallazgos),'ejemplos':ejemplos,
 'ier_ia':{'controles':controles,'cumplimiento_documental':sum(controles.values()),'total':len(controles),
 'nota':'Indicador documental preliminar; requiere evidencias y validación humana.'},
 'estado':'AUDITORIA_PRELIMINAR_NO_APTO_PARA_PRODUCCION'
}
archivo=EVID/'auditoria_preliminar_zenodo.json'
archivo.write_text(json.dumps(reporte,ensure_ascii=False,indent=2),encoding='utf-8')
print('Evidencia creada:',archivo)
print('IER-IA documental:',sum(controles.values()),'/',len(controles))
print('Siguiente etapa: contrastar imagePath, verificar imágenes reales, confirmar particiones y licencias antes de entrenar.')

In [ ]:

# ============================================================
# CELDA 8 - Diagnóstico de referencias y nombres de imágenes
# Actividad 5 | Control de acceso vehicular con IA
# ============================================================

from zipfile import ZipFile
from pathlib import PurePosixPath
from collections import Counter
import json

with ZipFile(zip_path, "r") as archivo_zip:
    nombres = set(archivo_zip.namelist())
    anotaciones = [
        n for n in nombres if n.lower().endswith(".json")
    ]

    resultados = Counter()
    ejemplos = []

    for ruta_json in sorted(anotaciones):
        dato = json.loads(
            archivo_zip.read(ruta_json).decode("utf-8-sig")
        )

        carpeta = PurePosixPath(ruta_json).parent
        imagen_declarada = str(dato.get("imagePath", "")).strip()

        # Comparar la ruta declarada con el archivo real.
        ruta_declarada = str(carpeta / imagen_declarada)
        ruta_por_nombre = str(
            PurePosixPath(ruta_json).with_suffix(".jpg")
        )

        existe_declarada = ruta_declarada in nombres
        existe_mismo_nombre = ruta_por_nombre in nombres

        if existe_declarada:
            resultados["referencia_correcta"] += 1
        elif existe_mismo_nombre:
            resultados["posible_error_de_nombre"] += 1
        else:
            resultados["sin_coincidencia"] += 1

        if not existe_declarada and len(ejemplos) < 12:
            ejemplos.append({
                "JSON": ruta_json,
                "imagePath": imagen_declarada,
                "JPG_mismo_nombre": existe_mismo_nombre
            })

print("=== DIAGNÓSTICO DE CORRESPONDENCIA ===")
print("JSON analizados:", len(anotaciones))
print("Resultados:", dict(resultados))

print("\n=== EJEMPLOS DE DISCREPANCIAS ===")
for ejemplo in ejemplos:
    print(ejemplo)

print("\nNOTA ÉTICA:")
print("No se modificaron etiquetas ni imágenes.")
print("Las discrepancias requieren validación antes de entrenar.")


In [ ]:

# ============================================================
# CELDA 9 - Validación visual de imágenes y anotaciones
# Actividad 5 | Control de acceso vehicular
# IA responsable: inspección sin modificar datos originales
# ============================================================

import json
import io
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from zipfile import ZipFile
from pathlib import PurePosixPath

ejemplos = [
    "00000.json",
    "00016.json",
    "00022.json",
    "00024.json",
    "00035.json",
    "00039.json"
]

with ZipFile(zip_path, "r") as archivo_zip:
    nombres = set(archivo_zip.namelist())

    fig, axes = plt.subplots(
        len(ejemplos), 2,
        figsize=(13, 4 * len(ejemplos))
    )

    for fila, nombre_json in enumerate(ejemplos):
        ruta_json = "UC3M-LP/test/" + nombre_json
        dato = json.loads(
            archivo_zip.read(ruta_json).decode("utf-8-sig")
        )

        carpeta = PurePosixPath(ruta_json).parent

        rutas = [
            str(carpeta / dato["imagePath"]),
            str(PurePosixPath(ruta_json).with_suffix(".jpg"))
        ]

        for columna, ruta in enumerate(rutas):
            ax = axes[fila, columna]

            if ruta not in nombres:
                ax.text(
                    0.5, 0.5, "Imagen no encontrada",
                    ha="center", va="center"
                )
                ax.axis("off")
                continue

            imagen = Image.open(
                io.BytesIO(archivo_zip.read(ruta))
            ).convert("RGB")

            dibujo = ImageDraw.Draw(imagen)

            for matricula in dato.get("lps", []):
                for caracter in matricula.get("characters", []):
                    bbox = caracter.get("bbox_coord", [])

                    if len(bbox) == 2:
                        try:
                            x1, y1 = bbox[0]
                            x2, y2 = bbox[1]

                            dibujo.rectangle(
                                [x1, y1, x2, y2],
                                outline="red",
                                width=8
                            )
                        except (TypeError, ValueError):
                            pass

            ax.imshow(imagen)
            ax.set_title(
                ("Declarada: " if columna == 0 else "Mismo nombre: ")
                + PurePosixPath(ruta).name
            )
            ax.axis("off")

    plt.tight_layout()
    plt.show()

print("Validación visual generada.")
print("No se modificaron imágenes ni anotaciones.")


In [ ]:

# ============================================================
# CELDA 10 - Validación estructural de imágenes y anotaciones
# Actividad 5 | Gestión de datos para entrenamiento
# IA responsable: preservar originales y registrar anomalías
# ============================================================

from zipfile import ZipFile
from pathlib import PurePosixPath
from PIL import Image
from collections import Counter
import json
import io
import pandas as pd

resultados = []
hallazgos = Counter()

with ZipFile(zip_path, "r") as archivo_zip:

    nombres = set(archivo_zip.namelist())

    archivos_json = sorted(
        n for n in nombres if n.lower().endswith(".json")
    )

    for ruta_json in archivos_json:

        registro = {
            "json": ruta_json,
            "imagen": "",
            "imagen_existe": False,
            "dimensiones_coinciden": False,
            "coordenadas_validas": False,
            "estado": "REVISAR"
        }

        try:
            dato = json.loads(
                archivo_zip.read(ruta_json).decode("utf-8-sig")
            )

            ruta_imagen = str(
                PurePosixPath(ruta_json).with_suffix(".jpg")
            )

            registro["imagen"] = ruta_imagen

            if ruta_imagen not in nombres:
                hallazgos["imagen_ausente"] += 1
                resultados.append(registro)
                continue

            registro["imagen_existe"] = True

            imagen = Image.open(
                io.BytesIO(archivo_zip.read(ruta_imagen))
            )

            ancho, alto = imagen.size

            registro["dimensiones_coinciden"] = (
                ancho == int(dato["imageWidth"])
                and alto == int(dato["imageHeight"])
            )

            coordenadas_correctas = True

            for matricula in dato.get("lps", []):
                for caracter in matricula.get("characters", []):

                    bbox = caracter.get("bbox_coord", [])

                    if len(bbox) != 2:
                        coordenadas_correctas = False
                        continue

                    try:
                        (x1, y1), (x2, y2) = bbox

                        if not (
                            0 <= x1 < x2 <= ancho
                            and 0 <= y1 < y2 <= alto
                        ):
                            coordenadas_correctas = False

                    except (TypeError, ValueError):
                        coordenadas_correctas = False

            registro["coordenadas_validas"] = coordenadas_correctas

            if (
                registro["imagen_existe"]
                and registro["dimensiones_coinciden"]
                and registro["coordenadas_validas"]
            ):
                registro["estado"] = "VALIDO_ESTRUCTURALMENTE"
                hallazgos["validos"] += 1
            else:
                hallazgos["requieren_revision"] += 1

        except Exception as error:
            hallazgos["errores_lectura"] += 1
            registro["detalle_error"] = str(error)[:150]

        resultados.append(registro)

df_validacion = pd.DataFrame(resultados)

# Guardar evidencia reproducible.
archivo_salida = EVID / "validacion_pares_uc3mlp.csv"

df_validacion.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

print("=== VALIDACIÓN ESTRUCTURAL ===")
print("Pares analizados:", len(df_validacion))
print("Resultados:", dict(hallazgos))
print("\nEstados:")
print(df_validacion["estado"].value_counts())
print("\nEvidencia guardada:", archivo_salida)

print("\nIA RESPONSABLE:")
print("Los datos originales no fueron modificados.")
print("La validación estructural no sustituye la revisión humana.")


In [ ]:

# ============================================================
# CELDA 11 - Análisis exploratorio y calidad del dataset
# Actividad 5 | Control de acceso vehicular
# IA responsable: distribución, representatividad y trazabilidad
# ============================================================

import json
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import PurePosixPath
from zipfile import ZipFile

registros = []
conteo = Counter()

with ZipFile(zip_path, "r") as archivo_zip:

    archivos_json = sorted(
        n for n in archivo_zip.namelist()
        if n.lower().endswith(".json")
    )

    for ruta_json in archivos_json:

        dato = json.loads(
            archivo_zip.read(ruta_json).decode("utf-8-sig")
        )

        particion = PurePosixPath(ruta_json).parts[1]

        for indice, matricula in enumerate(dato.get("lps", [])):

            for caracter in matricula.get("characters", []):

                etiqueta = str(caracter.get("char_id", ""))
                bbox = caracter.get("bbox_coord", [])

                if len(bbox) != 2:
                    continue

                (x1, y1), (x2, y2) = bbox

                registros.append({
                    "archivo_json": ruta_json,
                    "particion": particion,
                    "matricula": indice,
                    "caracter": etiqueta,
                    "ancho": x2 - x1,
                    "alto": y2 - y1
                })

                conteo[etiqueta] += 1

df_caracteres = pd.DataFrame(registros)

print("=== RESUMEN DEL DATASET ===")
print("Caracteres registrados:", len(df_caracteres))
print("Clases encontradas:", df_caracteres["caracter"].nunique())

print("\nDistribución por partición:")
print(df_caracteres["particion"].value_counts())

print("\nEstadísticas de dimensiones:")
print(df_caracteres[["ancho", "alto"]].describe())

print("\nClases menos representadas:")
print(df_caracteres["caracter"].value_counts().tail(10))

# Gráfica 1 - Distribución de caracteres
frecuencias = df_caracteres["caracter"].value_counts().sort_index()

plt.figure(figsize=(14, 5))
frecuencias.plot(kind="bar")
plt.title("Distribución de caracteres - UC3M-LP")
plt.xlabel("Carácter")
plt.ylabel("Número de ejemplos")
plt.tight_layout()
plt.show()

# Gráfica 2 - Dimensiones de las anotaciones
plt.figure(figsize=(9, 5))
plt.hist(df_caracteres["ancho"], bins=35, alpha=0.7, label="Ancho")
plt.hist(df_caracteres["alto"], bins=35, alpha=0.7, label="Alto")
plt.title("Distribución de dimensiones de caracteres")
plt.xlabel("Píxeles")
plt.ylabel("Frecuencia")
plt.legend()
plt.tight_layout()
plt.show()

# Guardar evidencia
ruta_salida = EVID / "analisis_exploratorio_caracteres.csv"
df_caracteres.to_csv(ruta_salida, index=False)

print("\nEvidencia guardada:", ruta_salida)
print("IA RESPONSABLE: distribución documentada; sesgos aún por evaluar.")


In [ ]:

# ============================================================
# CELDA 12 - Diagnóstico de clases y viabilidad del entrenamiento
# Actividad 5 | Gestión de datos para entrenamiento
#
# Objetivo:
# Evaluar qué clases cuentan con suficientes ejemplos
# para una validación cruzada de cinco particiones.
#
# IA RESPONSABLE:
# No eliminar ni duplicar clases sin documentar la decisión.
# ============================================================

import pandas as pd

# 1. Contabilizar caracteres por partición y clase.
tabla = pd.crosstab(
    df_caracteres["caracter"],
    df_caracteres["particion"]
)

# 2. Garantizar columnas esperadas.
for columna in ["train", "test"]:
    if columna not in tabla.columns:
        tabla[columna] = 0

# 3. Calcular total y viabilidad preliminar.
tabla["total"] = tabla["train"] + tabla["test"]

tabla["apta_cv5_por_conteo"] = tabla["train"] >= 5

tabla["requiere_revision"] = tabla["train"] < 5

# 4. Ordenar por menor representación.
tabla = tabla.sort_values("train")

print("=== DIAGNÓSTICO DE CLASES ===")
print(tabla.to_string())

print("\n=== RESUMEN ===")
print("Clases totales:", len(tabla))
print(
    "Clases con al menos 5 ejemplos de entrenamiento:",
    tabla["apta_cv5_por_conteo"].sum()
)
print(
    "Clases con menos de 5 ejemplos de entrenamiento:",
    tabla["requiere_revision"].sum()
)

print("\n=== CLASES QUE REQUIEREN REVISIÓN ===")
print(tabla[tabla["requiere_revision"]].to_string())

# 5. Guardar evidencia reproducible.
ruta = EVID / "diagnostico_clases_cv5.csv"

tabla.to_csv(ruta, encoding="utf-8-sig")

print("\nEvidencia guardada:", ruta)

print("\nIA RESPONSABLE:")
print("No se eliminaron, modificaron ni duplicaron ejemplos.")
print("La viabilidad final dependerá también de grupos independientes.")


In [ ]:

# ============================================================
# CELDA 13 - Manifiesto de preparación y exclusiones
# Actividad 5 | Control de acceso vehicular
#
# Objetivo:
# Identificar los caracteres incluidos en el experimento
# y documentar las exclusiones justificadas.
#
# IA RESPONSABLE:
# Preservar los datos originales y declarar las limitaciones
# de cobertura del modelo.
# ============================================================

import pandas as pd
import json
from datetime import datetime, timezone

# 1. Definir clases elegibles por conteo.
clases_elegibles = tabla.index[
    tabla["apta_cv5_por_conteo"]
].tolist()

clases_excluidas = tabla.index[
    ~tabla["apta_cv5_por_conteo"]
].tolist()

# 2. Crear manifiesto sin modificar datos originales.
df_manifiesto = df_caracteres.copy()

df_manifiesto["incluido_experimento"] = (
    df_manifiesto["caracter"].isin(clases_elegibles)
)

df_manifiesto["motivo_exclusion"] = ""

df_manifiesto.loc[
    ~df_manifiesto["incluido_experimento"],
    "motivo_exclusion"
] = "Menos de 5 ejemplos en entrenamiento"

# 3. Resumir selección.
print("=== PREPARACIÓN DEL EXPERIMENTO ===")
print("Clases elegibles:", len(clases_elegibles))
print("Clases excluidas:", clases_excluidas)

print("\nRegistros totales:", len(df_manifiesto))
print(
    "Registros incluidos:",
    df_manifiesto["incluido_experimento"].sum()
)
print(
    "Registros excluidos:",
    (~df_manifiesto["incluido_experimento"]).sum()
)

print("\nDistribución por partición:")
print(pd.crosstab(
    df_manifiesto["particion"],
    df_manifiesto["incluido_experimento"]
))

# 4. Guardar manifiesto reproducible.
ruta_csv = EVID / "manifiesto_preparacion.csv"

df_manifiesto.to_csv(
    ruta_csv,
    index=False,
    encoding="utf-8-sig"
)

# 5. Documentar la decisión técnica y ética.
decision = {
    "fecha_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "UC3M-LP",
    "criterio": "Minimo 5 ejemplos de entrenamiento por clase",
    "clases_elegibles": clases_elegibles,
    "clases_excluidas": clases_excluidas,
    "justificacion": (
        "Las clases excluidas no permiten CV de 5 particiones "
        "por insuficiencia de ejemplos de entrenamiento."
    ),
    "limitacion": (
        "El experimento no evaluara la capacidad de reconocer "
        "las clases excluidas."
    ),
    "datos_originales_modificados": False,
    "estado": "SELECCION_PRELIMINAR_PENDIENTE_DE_VALIDACION_POR_GRUPOS"
}

ruta_json = EVID / "decision_preparacion.json"

ruta_json.write_text(
    json.dumps(decision, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\nEvidencias generadas:")
print(ruta_csv)
print(ruta_json)

print("\nIA RESPONSABLE:")
print("Exclusiones documentadas y datos originales preservados.")
print("Pendiente: validación de independencia entre grupos.")


In [ ]:

# ============================================================
# CELDA 14 - Auditoría de grupos y posibles fugas de datos
# Actividad 5 | Control de acceso vehicular
#
# Objetivo:
# Comprobar si existen matrículas repetidas entre train/test
# y cuántos grupos diferentes tenemos por carácter.
#
# IA RESPONSABLE:
# Evitar conclusiones engañosas por fuga de información.
# No modificar el dataset original.
# ============================================================

from zipfile import ZipFile
from pathlib import PurePosixPath
from collections import Counter
import pandas as pd
import json

registros_grupos = []

with ZipFile(zip_path, "r") as archivo_zip:

    archivos_json = sorted(
        n for n in archivo_zip.namelist()
        if n.lower().endswith(".json")
    )

    for ruta_json in archivos_json:

        dato = json.loads(
            archivo_zip.read(ruta_json).decode("utf-8-sig")
        )

        particion = PurePosixPath(ruta_json).parts[1]

        for indice, matricula in enumerate(dato.get("lps", [])):

            identificador = str(
                matricula.get("lp_id", "")
            ).strip().upper()

            # Grupo de imagen/matrícula.
            grupo = f"{ruta_json}::matricula_{indice}"

            for caracter in matricula.get("characters", []):

                etiqueta = str(caracter.get("char_id", ""))

                if etiqueta not in clases_elegibles:
                    continue

                registros_grupos.append({
                    "particion": particion,
                    "archivo_json": ruta_json,
                    "grupo": grupo,
                    "matricula_id": identificador,
                    "caracter": etiqueta
                })

df_grupos = pd.DataFrame(registros_grupos)

print("=== AUDITORÍA DE GRUPOS ===")
print("Caracteres analizados:", len(df_grupos))
print("Grupos imagen-matrícula:", df_grupos["grupo"].nunique())

# 1. Matrículas repetidas entre train y test.
# Los identificadores vacíos se excluyen de este análisis.
df_identificados = df_grupos[
    df_grupos["matricula_id"].ne("")
].copy()

train_ids = set(
    df_identificados.loc[
        df_identificados["particion"] == "train",
        "matricula_id"
    ]
)

test_ids = set(
    df_identificados.loc[
        df_identificados["particion"] == "test",
        "matricula_id"
    ]
)

coincidencias = sorted(train_ids & test_ids)

print("\nIdentificadores compartidos train/test:",
      len(coincidencias))
print("Ejemplos:", coincidencias[:10])

# 2. Cantidad de grupos diferentes por clase en train.
grupos_por_clase = (
    df_grupos[df_grupos["particion"] == "train"]
    .groupby("caracter")["grupo"]
    .nunique()
    .sort_values()
)

print("\n=== GRUPOS DE ENTRENAMIENTO POR CLASE ===")
print(grupos_por_clase.to_string())

print("\nClases con menos de 5 grupos:")
print(grupos_por_clase[grupos_por_clase < 5])

# 3. Guardar evidencias sin publicar matrículas completas.
ruta_resumen = EVID / "grupos_por_clase.csv"

grupos_por_clase.to_csv(
    ruta_resumen,
    header=["grupos_independientes"],
    encoding="utf-8-sig"
)

resumen = {
    "caracteres_analizados": int(len(df_grupos)),
    "grupos_imagen_matricula": int(df_grupos["grupo"].nunique()),
    "identificadores_compartidos_train_test": len(coincidencias),
    "clases_con_menos_de_5_grupos": (
        grupos_por_clase[grupos_por_clase < 5].index.tolist()
    ),
    "nota": (
        "Auditoria preliminar. El identificador de matricula "
        "no equivale necesariamente a un identificador de vehiculo. "
        "No se publican identificadores completos."
    )
}

ruta_json = EVID / "auditoria_independencia.json"

ruta_json.write_text(
    json.dumps(resumen, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\nEvidencias generadas:")
print(ruta_resumen)
print(ruta_json)

print("\nIA RESPONSABLE:")
print("No se modificaron datos originales.")
print("No publicar identificadores completos de matrículas.")
print("Pendiente evaluar independencia real entre vehículos.")


In [ ]:

# ============================================================
# CELDA 15 - Diagnóstico de identificadores y fuga de datos
# Actividad 5 | Control de acceso vehicular
#
# IA RESPONSABLE:
# No publicar matrículas ni identificadores individuales.
# ============================================================

import pandas as pd
import json

datos = df_grupos.copy()

# Un registro por matrícula anotada.
placas = datos[
    ["particion", "grupo", "matricula_id"]
].drop_duplicates().copy()

placas["id_vacio"] = placas["matricula_id"].eq("")
placas["id_oculto"] = placas["matricula_id"].str.contains(
    "*", regex=False, na=False
)

# Número de grupos por identificador.
validos = placas[~placas["id_vacio"]]

frecuencias = validos.groupby(
    "matricula_id"
)["grupo"].nunique()

repetidos = frecuencias[frecuencias > 1]

# Identificadores presentes en ambas particiones.
cruzados = validos.groupby(
    "matricula_id"
)["particion"].nunique()

cruzados = cruzados[cruzados > 1]

# Evaluar grupos distintos por clase dentro de train.
train = datos[
    (datos["particion"] == "train")
    & (datos["matricula_id"] != "")
]

grupos_clase = train.groupby(
    "caracter"
)["matricula_id"].nunique().sort_values()

print("=== DIAGNÓSTICO DE IDENTIFICADORES ===")
print("Grupos imagen-matrícula:", len(placas))
print("Identificadores vacíos:", placas["id_vacio"].sum())
print("Identificadores ocultos:", placas["id_oculto"].sum())
print("Identificadores únicos:", len(frecuencias))
print("Identificadores repetidos:", len(repetidos))
print("Identificadores compartidos train/test:", len(cruzados))

print("\n=== INDEPENDENCIA POR CLASE ===")
print(grupos_clase.to_string())

print("\nClases con menos de cinco identificadores:")
print(grupos_clase[grupos_clase < 5].to_string())

# Guardar únicamente estadísticas agregadas.
resumen = {
    "grupos_imagen_matricula": int(len(placas)),
    "identificadores_vacios": int(placas["id_vacio"].sum()),
    "identificadores_ocultos": int(placas["id_oculto"].sum()),
    "identificadores_unicos": int(len(frecuencias)),
    "identificadores_repetidos": int(len(repetidos)),
    "identificadores_compartidos_train_test": int(len(cruzados)),
    "clases_menos_cinco_ids": grupos_clase[
        grupos_clase < 5
    ].index.tolist(),
    "advertencia": (
        "Los identificadores parcialmente ocultos pueden "
        "agrupar matrículas diferentes o no identificar "
        "todas las imágenes del mismo vehículo."
    )
}

salida = EVID / "diagnostico_identificadores.json"

salida.write_text(
    json.dumps(resumen, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nEvidencia generada:", salida)
print("IA RESPONSABLE: resultados agregados, sin matrículas.")
print("Estado: pendiente de decidir partición definitiva.")


In [ ]:

# ============================================================
# CELDA 16 - Evaluar partición por grupos
# Actividad 5 | Control de acceso vehicular
#
# Objetivo:
# Probar una separación reproducible por identificadores,
# comprobar solapamientos y cobertura de clases.
#
# IA RESPONSABLE:
# No modificar datos originales ni divulgar matrículas.
# ============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
import json

# 1. Trabajar únicamente con clases elegibles.
datos = df_grupos[
    df_grupos["caracter"].isin(clases_elegibles)
].copy().reset_index(drop=True)

# 2. Definir grupos conservadores por identificador.
# Advertencia: los identificadores están parcialmente ocultos.
datos["grupo_cv"] = datos["matricula_id"]

# 3. Proponer división 80/20 por grupos.
separador = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

indices_train, indices_test = next(
    separador.split(
        datos,
        y=datos["caracter"],
        groups=datos["grupo_cv"]
    )
)

train_exp = datos.iloc[indices_train].copy()
test_exp = datos.iloc[indices_test].copy()

# 4. Verificar solapamientos.
grupos_train = set(train_exp["grupo_cv"])
grupos_test = set(test_exp["grupo_cv"])

solapamiento = grupos_train & grupos_test

print("=== PARTICIÓN EXPERIMENTAL ===")
print("Train:", len(train_exp))
print("Test:", len(test_exp))
print("Grupos compartidos:", len(solapamiento))

# 5. Revisar representación por clase.
distribucion = pd.DataFrame({
    "train": train_exp["caracter"].value_counts(),
    "test": test_exp["caracter"].value_counts()
}).fillna(0).astype(int)

distribucion["total"] = (
    distribucion["train"] + distribucion["test"]
)

print("\n=== DISTRIBUCIÓN DE CLASES ===")
print(distribucion.sort_values("train").to_string())

print("\nClases ausentes en entrenamiento:")
print(distribucion[distribucion["train"] == 0].index.tolist())

print("\nClases ausentes en prueba:")
print(distribucion[distribucion["test"] == 0].index.tolist())

# 6. Guardar evidencia agregada.
ruta = EVID / "diagnostico_particion_grupos.csv"

distribucion.to_csv(ruta, encoding="utf-8-sig")

print("\nEvidencia generada:", ruta)
print("IA RESPONSABLE: identificadores no publicados.")
print("La partición es preliminar, no definitiva.")


In [ ]:

# ============================================================
# CELDA 17 - Validar StratifiedGroupKFold
# Actividad 5 | Control de acceso vehicular con IA
#
# OBJETIVO:
# Comprobar si la partición experimental permite
# validación cruzada estratificada por grupos.
#
# IA RESPONSABLE:
# Evitar fuga de información y documentar limitaciones.
# No se entrenarán modelos en esta celda.
# ============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# Usamos únicamente la partición de entrenamiento.
X_indices = np.arange(len(train_exp))
y = train_exp["caracter"].to_numpy()
grupos = train_exp["grupo_cv"].to_numpy()

cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

resultados_cv = []
clases_esperadas = set(clases_elegibles)

for fold, (idx_train, idx_val) in enumerate(
    cv.split(X_indices, y, groups=grupos), start=1
):

    clases_train = set(y[idx_train])
    clases_val = set(y[idx_val])

    grupos_train = set(grupos[idx_train])
    grupos_val = set(grupos[idx_val])

    compartidos = grupos_train & grupos_val

    registro = {
        "fold": fold,
        "registros_train": len(idx_train),
        "registros_validacion": len(idx_val),
        "grupos_compartidos": len(compartidos),
        "clases_train": len(clases_train),
        "clases_validacion": len(clases_val),
        "faltantes_train": sorted(
            clases_esperadas - clases_train
        ),
        "faltantes_validacion": sorted(
            clases_esperadas - clases_val
        )
    }

    resultados_cv.append(registro)

    print(f"\n=== FOLD {fold} ===")
    print("Train:", len(idx_train))
    print("Validación:", len(idx_val))
    print("Grupos compartidos:", len(compartidos))
    print("Clases ausentes en train:", registro["faltantes_train"])
    print(
        "Clases ausentes en validación:",
        registro["faltantes_validacion"]
    )

# Resumen general.
df_cv = pd.DataFrame(resultados_cv)

print("\n=== RESUMEN DE VALIDACIÓN CRUZADA ===")
print(df_cv.to_string(index=False))

# Guardar evidencia.
ruta_cv = EVID / "validacion_cruzada_grupos.csv"

df_cv.to_csv(
    ruta_cv,
    index=False,
    encoding="utf-8-sig"
)

print("\nEvidencia generada:", ruta_cv)

# Condiciones para aprobar la propuesta.
sin_fugas = all(
    r["grupos_compartidos"] == 0
    for r in resultados_cv
)

cobertura_completa = all(
    not r["faltantes_train"]
    and not r["faltantes_validacion"]
    for r in resultados_cv
)

print("\n=== DECISIÓN PRELIMINAR ===")
print("Sin grupos compartidos:", sin_fugas)
print("Cobertura completa de clases:", cobertura_completa)

if sin_fugas and cobertura_completa:
    print("CV3 viable según las comprobaciones realizadas.")
else:
    print("CV3 requiere ajustes antes del entrenamiento.")

print("\nIA RESPONSABLE:")
print("No se modificaron ni duplicaron registros.")
print("La independencia real entre vehículos sigue siendo una limitación.")


In [ ]:

# ============================================================
# CELDA 18 - Ajuste de clases y validación cruzada definitiva
# Actividad 5 | Control de acceso vehicular
#
# Objetivo:
# Excluir clases insuficientemente representadas y comprobar
# la viabilidad de CV3 mediante StratifiedGroupKFold.
#
# IA RESPONSABLE:
# Documentar exclusiones y preservar datos originales.
# ============================================================

import numpy as np
import pandas as pd
import json
from sklearn.model_selection import StratifiedGroupKFold

# 1. Exclusiones justificadas.
clases_excluidas_final = ["I", "U", "O"]

clases_finales = sorted(
    set(df_grupos["caracter"]) -
    set(clases_excluidas_final)
)

# 2. Construir conjunto experimental.
datos_finales = df_grupos[
    df_grupos["caracter"].isin(clases_finales)
].copy()

train_final = datos_finales[
    datos_finales["grupo_cv"].notna()
].copy() if "grupo_cv" in datos_finales.columns else None

# Conservar los índices de la partición experimental
# definida en la CELDA 16.
train_final = train_exp[
    train_exp["caracter"].isin(clases_finales)
].copy().reset_index(drop=True)

test_final = test_exp[
    test_exp["caracter"].isin(clases_finales)
].copy().reset_index(drop=True)

# 3. Configurar validación cruzada.
cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

y = train_final["caracter"].to_numpy()
grupos = train_final["grupo_cv"].to_numpy()
indices = np.arange(len(train_final))

resultados = []

for fold, (idx_train, idx_val) in enumerate(
    cv.split(indices, y, groups=grupos), start=1
):

    grupos_compartidos = set(grupos[idx_train]) & set(
        grupos[idx_val]
    )

    faltantes_train = sorted(
        set(clases_finales) - set(y[idx_train])
    )

    faltantes_val = sorted(
        set(clases_finales) - set(y[idx_val])
    )

    resultados.append({
        "fold": fold,
        "train": len(idx_train),
        "validacion": len(idx_val),
        "grupos_compartidos": len(grupos_compartidos),
        "faltantes_train": faltantes_train,
        "faltantes_validacion": faltantes_val
    })

    print(f"\n=== FOLD {fold} ===")
    print("Entrenamiento:", len(idx_train))
    print("Validación:", len(idx_val))
    print("Grupos compartidos:", len(grupos_compartidos))
    print("Clases ausentes train:", faltantes_train)
    print("Clases ausentes validación:", faltantes_val)

# 4. Comprobar viabilidad.
cv_viable = all(
    r["grupos_compartidos"] == 0
    and not r["faltantes_train"]
    and not r["faltantes_validacion"]
    for r in resultados
)

print("\n=== RESUMEN ===")
print("Clases finales:", len(clases_finales))
print("Caracteres train:", len(train_final))
print("Caracteres test:", len(test_final))
print("CV3 viable:", cv_viable)

# 5. Guardar evidencia.
resumen = {
    "clases_excluidas": clases_excluidas_final,
    "clases_utilizadas": clases_finales,
    "numero_clases": len(clases_finales),
    "train": len(train_final),
    "test": len(test_final),
    "resultados_cv": resultados,
    "cv_viable": bool(cv_viable),
    "advertencia": (
        "La ausencia de grupos compartidos se comprueba "
        "sobre identificadores parcialmente anonimizados."
    )
}

ruta = EVID / "validacion_cv3_final.json"

ruta.write_text(
    json.dumps(resumen, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nEvidencia generada:", ruta)
print("IA RESPONSABLE: exclusiones registradas.")
print("Los datos originales permanecen intactos.")


In [ ]:

# ============================================================
# CELDA 19 - Extracción y normalización CORREGIDA
# Actividad 5 | Control de acceso vehicular
#
# OBJETIVO:
# Generar exactamente un registro por carácter seleccionado.
#
# IA RESPONSABLE:
# Preservar los originales, evitar duplicaciones y
# conservar la trazabilidad de cada observación.
# ============================================================

import io
import json
import numpy as np
from PIL import Image
from zipfile import ZipFile
from pathlib import PurePosixPath
from tqdm.auto import tqdm

TAMANO = (32, 32)

# Liberar los arreglos generados por la versión anterior.
for variable in ["X", "y", "particiones", "grupos_ml"]:
    if variable in globals():
        del globals()[variable]

# Crear un índice de los grupos y su partición.
mapa_grupos = {}

for nombre_conjunto, tabla_datos in [
    ("train", train_final),
    ("test", test_final)
]:
    for _, fila in tabla_datos[
        ["archivo_json", "grupo", "grupo_cv"]
    ].drop_duplicates().iterrows():

        clave = (fila["archivo_json"], fila["grupo"])
        mapa_grupos[clave] = (
            nombre_conjunto,
            fila["grupo_cv"]
        )

imagenes = []
etiquetas = []
conjuntos = []
grupos_modelo = []
errores = []

with ZipFile(zip_path, "r") as archivo_zip:

    archivos_json = sorted(set(
        ruta for ruta, grupo in mapa_grupos
    ))

    for ruta_json in tqdm(
        archivos_json,
        desc="Extrayendo caracteres"
    ):
        try:
            dato = json.loads(
                archivo_zip.read(ruta_json).decode("utf-8-sig")
            )

            ruta_imagen = str(
                PurePosixPath(ruta_json).with_suffix(".jpg")
            )

            with Image.open(
                io.BytesIO(archivo_zip.read(ruta_imagen))
            ) as origen:
                imagen = origen.convert("L")

            for indice, matricula in enumerate(
                dato.get("lps", [])
            ):
                grupo = f"{ruta_json}::matricula_{indice}"
                clave = (ruta_json, grupo)

                if clave not in mapa_grupos:
                    continue

                conjunto, grupo_cv = mapa_grupos[clave]

                for caracter in matricula.get("characters", []):

                    etiqueta = str(
                        caracter.get("char_id", "")
                    )

                    if etiqueta not in clases_finales:
                        continue

                    (x1, y1), (x2, y2) = caracter["bbox_coord"]

                    recorte = imagen.crop(
                        (int(x1), int(y1), int(x2), int(y2))
                    )

                    recorte = recorte.resize(TAMANO)

                    vector = np.asarray(
                        recorte, dtype=np.float32
                    ).flatten() / 255.0

                    imagenes.append(vector)
                    etiquetas.append(etiqueta)
                    conjuntos.append(conjunto)
                    grupos_modelo.append(grupo_cv)

        except Exception as error:
            errores.append({
                "archivo": ruta_json,
                "error": str(error)[:150]
            })

X = np.asarray(imagenes, dtype=np.float32)
y = np.asarray(etiquetas)
particiones = np.asarray(conjuntos)
grupos_ml = np.asarray(grupos_modelo)

# Verificaciones obligatorias.
esperado_train = len(train_final)
esperado_test = len(test_final)

obtenido_train = int(np.sum(particiones == "train"))
obtenido_test = int(np.sum(particiones == "test"))

assert len(errores) == 0, f"Errores: {errores[:5]}"
assert obtenido_train == esperado_train, (
    f"Train: esperado {esperado_train}, "
    f"obtenido {obtenido_train}"
)
assert obtenido_test == esperado_test, (
    f"Test: esperado {esperado_test}, "
    f"obtenido {obtenido_test}"
)
assert X.shape == (12744, 1024), (
    f"Dimensiones inesperadas: {X.shape}"
)

print("=== DATOS NORMALIZADOS ===")
print("Matriz X:", X.shape)
print("Etiquetas y:", y.shape)
print("Entrenamiento:", obtenido_train)
print("Prueba:", obtenido_test)
print("Errores:", len(errores))
print("Validación de cantidades: APROBADA")

print("\nIA RESPONSABLE:")
print("Sin duplicación por recorrido.")
print("Datos originales preservados.")
print("Particiones y grupos conservados.")



In [ ]:

# ============================================================
# CELDA 20 - Versionamiento del dataset limpio
# Actividad 5 | Gestión de datos para entrenamiento
#
# OBJETIVO:
# Guardar datos normalizados, verificar su consistencia
# y generar evidencias reproducibles.
#
# IA RESPONSABLE:
# Preservar la fuente y evitar publicar imágenes o
# matrículas identificables sin autorización.
# ============================================================

import numpy as np
import pandas as pd
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

VERSION = "v1.0.0"
LIMPIOS = RAIZ / "datos" / "datos_limp"
LIMPIOS.mkdir(parents=True, exist_ok=True)

# 1. Verificar correspondencia con el manifiesto.
esperado = pd.crosstab(
    df_manifiesto.loc[
        df_manifiesto["caracter"].isin(clases_finales),
        "particion"
    ],
    df_manifiesto.loc[
        df_manifiesto["caracter"].isin(clases_finales),
        "caracter"
    ]
)

# La partición experimental es distinta de la original;
# comprobamos el total por clase, no el reparto train/test.
conteo_original = (
    df_manifiesto[
        df_manifiesto["caracter"].isin(clases_finales)
    ]["caracter"].value_counts().sort_index()
)

conteo_procesado = (
    pd.Series(y).value_counts().sort_index()
)

assert conteo_original.equals(conteo_procesado), (
    "Las etiquetas procesadas no coinciden con el manifiesto."
)

assert len(X) == len(y) == len(particiones) == len(grupos_ml)
assert X.shape[1] == 1024
assert np.isfinite(X).all()
assert X.min() >= 0 and X.max() <= 1

# 2. Separar los datos ya preparados.
mascara_train = particiones == "train"
mascara_test = particiones == "test"

archivos = {
    "train": LIMPIOS / f"caracteres_train_{VERSION}.npz",
    "test": LIMPIOS / f"caracteres_test_{VERSION}.npz"
}

for nombre, mascara in [
    ("train", mascara_train),
    ("test", mascara_test)
]:
    np.savez_compressed(
        archivos[nombre],
        X=X[mascara],
        y=y[mascara],
        grupos=grupos_ml[mascara]
    )

# 3. Calcular hashes para verificar integridad.
def sha256_archivo(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()

hashes = {
    nombre: sha256_archivo(ruta)
    for nombre, ruta in archivos.items()
}

# 4. Crear manifiesto de versión.
manifiesto_version = {
    "version": VERSION,
    "fecha_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_fuente": "UC3M-LP",
    "zenodo_record_id": ZENODO_RECORD_ID,
    "semilla": SEED,
    "dimensiones_imagen": [32, 32],
    "normalizacion": "Escala de grises, valores 0-1",
    "clases": clases_finales,
    "clases_excluidas": ["I", "U", "O"],
    "registros_train": int(mascara_train.sum()),
    "registros_test": int(mascara_test.sum()),
    "hashes_sha256": hashes,
    "advertencia": (
        "Dataset derivado para uso experimental. "
        "Los identificadores de grupo proceden de "
        "matriculas parcialmente anonimizadas. "
        "No se garantiza independencia real entre vehiculos."
    ),
    "uso_responsable": (
        "No utilizar este modelo como autorizador automatico "
        "de acceso vehicular sin validacion adicional."
    )
}

ruta_manifiesto = (
    EVID / f"manifiesto_dataset_{VERSION}.json"
)

ruta_manifiesto.write_text(
    json.dumps(
        manifiesto_version,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("=== VERSIONAMIENTO COMPLETADO ===")
print("Versión:", VERSION)
print("Train:", archivos["train"])
print("Test:", archivos["test"])

print("\n=== INTEGRIDAD SHA-256 ===")
for nombre, valor in hashes.items():
    print(nombre, valor)

print("\nManifiesto:", ruta_manifiesto)
print("Clases:", len(clases_finales))
print("Registros:", len(X))
print("Estado: DATOS PREPARADOS Y VERSIONADOS")

print("\nIA RESPONSABLE:")
print("Originales preservados.")
print("Limitaciones y restricciones de uso documentadas.")


In [ ]:

# ============================================================
# CELDA 21 - Verificación final del dataset versionado
# Actividad 5 | Control de acceso vehicular
#
# OBJETIVO:
# Verificar integridad, cantidades, grupos y duplicados.
#
# IA RESPONSABLE:
# Detectar posibles fugas de información antes de entrenar.
# ============================================================

import numpy as np
import hashlib
import json
from collections import Counter

# 1. Verificar los hashes de los archivos guardados.
for nombre, ruta in archivos.items():
    actual = sha256_archivo(ruta)
    esperado = manifiesto_version["hashes_sha256"][nombre]
    assert actual == esperado, f"Hash incorrecto: {nombre}"

print("=== INTEGRIDAD ===")
print("SHA-256: archivos íntegros")

# 2. Cargar nuevamente los datos desde disco.
with np.load(archivos["train"], allow_pickle=False) as a:
    X_train = a["X"]
    y_train = a["y"]
    groups_train = a["grupos"]

with np.load(archivos["test"], allow_pickle=False) as a:
    X_test = a["X"]
    y_test = a["y"]
    groups_test = a["grupos"]

assert X_train.shape == (10178, 1024)
assert X_test.shape == (2566, 1024)

assert len(y_train) == len(groups_train) == 10178
assert len(y_test) == len(groups_test) == 2566

assert np.isfinite(X_train).all()
assert np.isfinite(X_test).all()

# 3. Verificar grupos compartidos.
compartidos = set(groups_train) & set(groups_test)

print("\n=== PARTICIONES ===")
print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Grupos compartidos:", len(compartidos))

# 4. Detectar recortes normalizados exactamente iguales.
# Usamos hashes de los vectores, sin exponer las imágenes.
def hashes_imagenes(matriz):
    return [
        hashlib.sha256(
            np.ascontiguousarray(fila).tobytes()
        ).hexdigest()
        for fila in matriz
    ]

hash_train = hashes_imagenes(X_train)
hash_test = hashes_imagenes(X_test)

conteo_train = Counter(hash_train)
conteo_test = Counter(hash_test)

duplicados_train = sum(
    cantidad - 1 for cantidad in conteo_train.values()
)
duplicados_test = sum(
    cantidad - 1 for cantidad in conteo_test.values()
)

hashes_compartidos = set(hash_train) & set(hash_test)

registros_train_compartidos = sum(
    conteo_train[h] for h in hashes_compartidos
)
registros_test_compartidos = sum(
    conteo_test[h] for h in hashes_compartidos
)

print("\n=== DUPLICADOS EXACTOS ===")
print("Duplicados internos train:", duplicados_train)
print("Duplicados internos test:", duplicados_test)
print("Patrones idénticos compartidos:", len(hashes_compartidos))
print("Registros train afectados:", registros_train_compartidos)
print("Registros test afectados:", registros_test_compartidos)

# 5. Determinar si podemos continuar.
aprobado = (
    len(compartidos) == 0
    and len(hashes_compartidos) == 0
)

reporte_21 = {
    "version_dataset": VERSION,
    "train": len(X_train),
    "test": len(X_test),
    "grupos_compartidos": len(compartidos),
    "duplicados_train": duplicados_train,
    "duplicados_test": duplicados_test,
    "patrones_compartidos": len(hashes_compartidos),
    "registros_train_afectados": registros_train_compartidos,
    "registros_test_afectados": registros_test_compartidos,
    "sin_fuga_exacta_detectada": aprobado,
    "limitacion": (
        "Las coincidencias exactas no detectan duplicados "
        "visualmente similares ni vehículos repetidos."
    )
}

salida = EVID / "validacion_final_dataset.json"

salida.write_text(
    json.dumps(reporte_21, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\n=== RESULTADO ===")
print("Sin solapamiento exacto detectado:", aprobado)
print("Evidencia guardada:", salida)

if not aprobado:
    print("REVISAR: No iniciar entrenamiento todavía.")
else:
    print("Control de duplicados cruzados superado.")
    print("Pendiente validar otros riesgos antes de producción.")

print("\nIA RESPONSABLE:")
print("No se publicaron imágenes ni matrículas.")
print("Se verificó integridad y posible duplicación exacta.")


In [ ]:

# ============================================================
# CELDA 22 - Configuración de MLflow (VERSIÓN CORREGIDA)
# Actividad 5 | Control de acceso vehicular
#
# OBJETIVO:
# Configurar MLflow utilizando SQLite para registrar
# experimentos, parámetros, métricas y modelos.
#
# IA RESPONSABLE:
# Garantizar trazabilidad y reproducibilidad.
# ============================================================

import sys
import subprocess
import importlib.util
import platform
from pathlib import Path

# 1. Verificar instalación de MLflow.
if importlib.util.find_spec("mlflow") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip",
        "install", "-q", "mlflow"
    ])

import mlflow
import sklearn
import numpy as np

# 2. Crear directorios de trabajo.
MLFLOW_DIR = RAIZ / "mlflow"
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)

# 3. Configurar SQLite como backend.
DB_PATH = MLFLOW_DIR / "mlflow.db"

TRACKING_URI = (
    f"sqlite:///{DB_PATH.resolve().as_posix()}"
)

mlflow.set_tracking_uri(TRACKING_URI)

# 4. Crear o seleccionar experimento.
NOMBRE_EXPERIMENTO = (
    "Actividad5_ControlAccesoVehicular_Clasificacion"
)

experimento = mlflow.set_experiment(
    NOMBRE_EXPERIMENTO
)

# 5. Mostrar información de configuración.
print("=== ENTORNO MLFLOW ===")
print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("MLflow:", mlflow.__version__)

print("\n=== CONFIGURACIÓN ===")
print("Experimento:", NOMBRE_EXPERIMENTO)
print("ID:", experimento.experiment_id)
print("Backend:", "SQLite")
print("Base de datos:", DB_PATH)
print("Tracking URI:", mlflow.get_tracking_uri())

print("\n=== DATOS DISPONIBLES ===")
print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Clases:", len(np.unique(y_train)))

# 6. Verificaciones.
assert DB_PATH.exists(), (
    "No se creó la base de datos SQLite."
)

assert X_train.shape == (10178, 1024)
assert X_test.shape == (2566, 1024)

print("\nMLFLOW CONFIGURADO CORRECTAMENTE")

print("\nIA RESPONSABLE:")
print("Seguimiento reproducible de experimentos.")
print("No se utilizarán modelos para autorizar accesos reales.")



In [ ]:

# ============================================================
# CELDA 23 - Entrenamiento de modelos base y MLflow
# Actividad 5 | Control de acceso vehicular
#
# OBJETIVO:
# Entrenar LinearSVC y Random Forest con configuraciones
# iniciales y registrar experimentos reales en MLflow.
#
# IA RESPONSABLE:
# No utilizar la partición de prueba para seleccionar modelos.
# Documentar parámetros, resultados y limitaciones.
# ============================================================

import time
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import sklearn

from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.model_selection import GroupShuffleSplit

# 1. Crear una validación preliminar separada por grupos.
# Se utiliza exclusivamente el conjunto de entrenamiento.
separador_base = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

idx_entrena, idx_valida = next(
    separador_base.split(
        X_train,
        y_train,
        groups=groups_train
    )
)

assert not (
    set(groups_train[idx_entrena])
    & set(groups_train[idx_valida])
), "Se detectaron grupos compartidos."

X_base_train = X_train[idx_entrena]
y_base_train = y_train[idx_entrena]

X_base_val = X_train[idx_valida]
y_base_val = y_train[idx_valida]

print("=== PARTICIÓN PARA MODELOS BASE ===")
print("Entrenamiento:", len(X_base_train))
print("Validación:", len(X_base_val))
print("Prueba final: reservada, sin utilizar")

# 2. Definir modelos iniciales.
modelos = {
    "LinearSVC_Base": LinearSVC(
        C=1.0,
        dual=False,
        max_iter=2000,
        random_state=42
    ),
    "RandomForest_Base": RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        random_state=42,
        n_jobs=-1
    )
}

resultados_base = []

# 3. Entrenar y registrar cada modelo.
for nombre, modelo in modelos.items():

    print(f"\n=== ENTRENANDO {nombre} ===")

    with mlflow.start_run(run_name=nombre):

        mlflow.set_tags({
            "actividad": "5",
            "proyecto": "Control de acceso vehicular",
            "etapa": "modelo_base",
            "dataset_version": VERSION,
            "uso": "experimental_no_produccion",
            "ia_responsable": "supervision_humana_requerida"
        })

        mlflow.log_params({
            "algoritmo": nombre,
            "semilla": 42,
            "caracteristicas": X_train.shape[1],
            "clases": len(clases_finales),
            "registros_entrenamiento": len(X_base_train),
            "registros_validacion": len(X_base_val)
        })

        mlflow.log_params(
            {
                f"modelo_{k}": v
                for k, v in modelo.get_params().items()
                if isinstance(v, (str, int, float, bool))
            }
        )

        inicio = time.perf_counter()
        modelo.fit(X_base_train, y_base_train)
        tiempo = time.perf_counter() - inicio

        predicciones = modelo.predict(X_base_val)

        metricas = {
            "accuracy": accuracy_score(
                y_base_val, predicciones
            ),
            "precision_macro": precision_score(
                y_base_val, predicciones,
                average="macro", zero_division=0
            ),
            "recall_macro": recall_score(
                y_base_val, predicciones,
                average="macro", zero_division=0
            ),
            "f1_macro": f1_score(
                y_base_val, predicciones,
                average="macro", zero_division=0
            ),
            "tiempo_entrenamiento_seg": tiempo
        }

        mlflow.log_metrics(metricas)

        # Registrar el modelo entrenado como artefacto.
        mlflow.sklearn.log_model(
            sk_model=modelo,
            name="modelo_entrenado"
        )

        resultados_base.append({
            "modelo": nombre,
            **metricas
        })

        print("Tiempo:", round(tiempo, 2), "segundos")
        print("Accuracy:", round(metricas["accuracy"], 4))
        print("F1 macro:", round(metricas["f1_macro"], 4))
        print("Registro MLflow: completado")

# 4. Generar tabla comparativa.
df_resultados_base = pd.DataFrame(resultados_base)

print("\n=== COMPARACIÓN DE MODELOS BASE ===")
print(df_resultados_base.to_string(index=False))

# 5. Guardar evidencia.
ruta_resultados = EVID / "resultados_modelos_base.csv"

df_resultados_base.to_csv(
    ruta_resultados,
    index=False,
    encoding="utf-8-sig"
)

print("\nEvidencia:", ruta_resultados)
print("Experimentos disponibles en MLflow.")

print("\nIA RESPONSABLE:")
print("Resultados experimentales, no operativos.")
print("El conjunto de prueba final no fue utilizado.")


In [ ]:

# ============================================================
# CELDA 24 - Recuperación del experimento Random Forest
# Actividad 5 | Control de acceso vehicular
#
# OBJETIVO:
# Recuperar el modelo entrenado y completar su registro
# utilizando una serialización compatible con MLflow.
#
# IA RESPONSABLE:
# Conservar métricas reales y evitar entrenamientos
# innecesarios o resultados inventados.
# ============================================================

import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. Recuperar modelo desde memoria.
modelo_rf = modelos["RandomForest_Base"]

# 2. Verificar que realmente esté entrenado.
assert hasattr(modelo_rf, "estimators_"), (
    "El modelo no está entrenado. "
    "Revisar ejecución de la CELDA 23."
)

# 3. Recuperar predicciones reales.
pred_rf = modelo_rf.predict(X_base_val)

metricas_rf = {
    "accuracy": accuracy_score(y_base_val, pred_rf),
    "precision_macro": precision_score(
        y_base_val, pred_rf,
        average="macro", zero_division=0
    ),
    "recall_macro": recall_score(
        y_base_val, pred_rf,
        average="macro", zero_division=0
    ),
    "f1_macro": f1_score(
        y_base_val, pred_rf,
        average="macro", zero_division=0
    )
}

print("=== RANDOM FOREST RECUPERADO ===")
for nombre, valor in metricas_rf.items():
    print(nombre, round(valor, 4))

# 4. Registrar modelo en una ejecución nueva.
with mlflow.start_run(
    run_name="RandomForest_Base_Recuperado"
):

    mlflow.set_tags({
        "actividad": "5",
        "etapa": "modelo_base",
        "dataset_version": VERSION,
        "estado": "recuperado_tras_error_serializacion",
        "uso": "experimental_no_produccion",
        "ia_responsable": "supervision_humana_requerida"
    })

    mlflow.log_params({
        "algoritmo": "RandomForest",
        "n_estimators": modelo_rf.n_estimators,
        "max_depth": modelo_rf.max_depth,
        "semilla": 42,
        "registros_train": len(X_base_train),
        "registros_val": len(X_base_val)
    })

    mlflow.log_metrics(metricas_rf)

    # Evitar el error previo de tipos no confiables.
    mlflow.sklearn.log_model(
        sk_model=modelo_rf,
        name="modelo_entrenado",
        serialization_format="cloudpickle"
    )

print("\n=== REGISTRO COMPLETADO ===")
print("Random Forest registrado en MLflow.")

print("\nIA RESPONSABLE:")
print("Métricas calculadas sobre predicciones reales.")
print("Modelo recuperado sin repetir entrenamiento.")
print("Datos de prueba final no utilizados.")


In [ ]:

# ============================================================
# CELDA 25 - Consolidación de resultados base desde MLflow
# Actividad 5 | Control de acceso vehicular
#
# OBJETIVO:
# Recuperar parámetros y métricas registrados en MLflow
# para construir la comparación inicial.
#
# IA RESPONSABLE:
# Utilizar únicamente resultados reales y verificables.
# ============================================================

import mlflow
import pandas as pd

# 1. Recuperar identificador del experimento.
experimento = mlflow.get_experiment_by_name(
    NOMBRE_EXPERIMENTO
)

assert experimento is not None, (
    "No se encontró el experimento en MLflow."
)

# 2. Consultar las ejecuciones registradas.
ejecuciones = mlflow.search_runs(
    experiment_ids=[experimento.experiment_id]
)

print("=== EJECUCIONES REGISTRADAS ===")
print("Total:", len(ejecuciones))

# 3. Seleccionar únicamente modelos base exitosos.
nombres_base = [
    "LinearSVC_Base",
    "RandomForest_Base_Recuperado"
]

ejecuciones_base = ejecuciones[
    (ejecuciones["tags.mlflow.runName"].isin(nombres_base))
    & (ejecuciones["status"] == "FINISHED")
].copy()

assert set(nombres_base).issubset(
    set(ejecuciones_base["tags.mlflow.runName"])
), "Falta registrar uno de los modelos base."

# 4. Preparar tabla comparativa.
columnas = {
    "tags.mlflow.runName": "modelo",
    "metrics.accuracy": "accuracy",
    "metrics.precision_macro": "precision_macro",
    "metrics.recall_macro": "recall_macro",
    "metrics.f1_macro": "f1_macro",
    "metrics.tiempo_entrenamiento_seg": "tiempo_segundos"
}

columnas_disponibles = [
    c for c in columnas if c in ejecuciones_base.columns
]

comparacion = ejecuciones_base[
    columnas_disponibles
].rename(columns=columnas)

comparacion = comparacion.drop_duplicates(
    subset=["modelo"], keep="first"
)

comparacion = comparacion.sort_values("modelo")

print("\n=== COMPARACIÓN DE MODELOS BASE ===")
print(comparacion.to_string(index=False))

# 5. Identificar mejor modelo preliminar por F1.
mejor = comparacion.sort_values(
    "f1_macro", ascending=False
).iloc[0]

print("\n=== RESULTADO PRELIMINAR ===")
print("Mayor F1 macro:", mejor["modelo"])
print("F1 macro:", round(mejor["f1_macro"], 4))

# 6. Guardar evidencia.
ruta = EVID / "comparacion_modelos_base_mlflow.csv"

comparacion.to_csv(
    ruta, index=False, encoding="utf-8-sig"
)

print("\nEvidencia generada:", ruta)

print("\nIA RESPONSABLE:")
print("Métricas extraídas de MLflow.")
print("Comparación preliminar, no decisión de producción.")


In [ ]:

# ============================================================
# CELDA 26 - Preparación de búsqueda de hiperparámetros
# Actividad 5 | Control de acceso vehicular
#
# OBJETIVO:
# Configurar GridSearchCV para LinearSVC y Random Forest
# utilizando validación cruzada estratificada por grupos.
#
# IA RESPONSABLE:
# No utilizar el conjunto de prueba para optimizar modelos.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.base import clone

# 1. Configurar CV reproducible.
CV_FINAL = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# 2. Conservar los pliegues para ambos modelos.
pliegues = list(
    CV_FINAL.split(
        X_train,
        y_train,
        groups=groups_train
    )
)

# 3. Comprobar ausencia de solapamientos.
clases_esperadas = set(clases_finales)

for numero, (idx_train, idx_val) in enumerate(
    pliegues, start=1
):
    grupos_a = set(groups_train[idx_train])
    grupos_b = set(groups_train[idx_val])

    assert not (grupos_a & grupos_b), (
        f"Grupos compartidos en fold {numero}"
    )

    assert set(y_train[idx_train]) == clases_esperadas, (
        f"Faltan clases de entrenamiento en fold {numero}"
    )

    assert set(y_train[idx_val]) == clases_esperadas, (
        f"Faltan clases de validación en fold {numero}"
    )

print("Validación cruzada: 3 pliegues comprobados.")

# 4. Definir modelos.
# El escalado de LinearSVC se ajustará dentro de cada fold.
modelos_grid = {
    "LinearSVC": Pipeline([
        ("escalado", StandardScaler()),
        ("modelo", LinearSVC(
            dual=False,
            max_iter=3000,
            random_state=42
        ))
    ]),
    "RandomForest": RandomForestClassifier(
        random_state=42,
        n_jobs=1
    )
}

# 5. Definir búsquedas controladas.
parametros_grid = {
    "LinearSVC": {
        "modelo__C": [0.1, 1.0]
    },
    "RandomForest": {
        "n_estimators": [80, 120],
        "max_depth": [15, 25]
    }
}

# 6. Calcular número de entrenamientos.
from sklearn.model_selection import ParameterGrid

total_estimado = 0

print("\n=== PLAN DE OPTIMIZACIÓN ===")

for nombre, parametros in parametros_grid.items():

    combinaciones = len(list(ParameterGrid(parametros)))
    entrenamientos = combinaciones * len(pliegues)

    total_estimado += entrenamientos

    print(f"\nModelo: {nombre}")
    print("Combinaciones:", combinaciones)
    print("Entrenamientos CV:", entrenamientos)
    print("Parámetros:", parametros)

print("\nTotal de entrenamientos CV:", total_estimado)

print("\n=== CONTROL DE DATOS ===")
print("Entrenamiento:", X_train.shape)
print("Prueba reservada:", X_test.shape)
print("Clases:", len(clases_finales))

print("\nIA RESPONSABLE:")
print("Se conserva el conjunto de prueba sin utilizar.")
print("La normalización de LinearSVC se ajusta dentro de CV.")
print("Se mantienen los grupos originales del experimento.")


In [ ]:

# ============================================================
# CELDA 27 - Optimización de modelos con GridSearchCV y MLflow
# Actividad 5 | Control de acceso vehicular con IA
#
# OBJETIVO:
# Optimizar LinearSVC y Random Forest mediante validación
# cruzada, registrar los experimentos y comparar resultados.
#
# IA RESPONSABLE:
# Mantener el conjunto de prueba reservado.
# Registrar resultados reales, decisiones y limitaciones.
# ============================================================

import time
import json
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.model_selection import GridSearchCV
from sklearn.base import clone

resultados_grid = []
modelos_optimizados = {}

for nombre, modelo in modelos_grid.items():

    print(f"\n=== OPTIMIZANDO {nombre} ===")

    busqueda = GridSearchCV(
        estimator=clone(modelo),
        param_grid=parametros_grid[nombre],
        scoring="f1_macro",
        cv=pliegues,
        refit=True,
        n_jobs=1,
        verbose=1,
        error_score="raise",
        return_train_score=True
    )

    inicio = time.perf_counter()

    # Se utiliza únicamente entrenamiento.
    busqueda.fit(X_train, y_train)

    duracion = time.perf_counter() - inicio

    modelos_optimizados[nombre] = busqueda.best_estimator_

    # Registrar resultados de todas las combinaciones.
    tabla_cv = pd.DataFrame(busqueda.cv_results_)

    ruta_tabla = (
        EVID / f"gridsearch_{nombre.lower()}_cv.csv"
    )

    tabla_cv.to_csv(
        ruta_tabla,
        index=False,
        encoding="utf-8-sig"
    )

    # Crear registro de experimento.
    with mlflow.start_run(
        run_name=f"{nombre}_GridSearchCV"
    ):

        mlflow.set_tags({
            "actividad": "5",
            "etapa": "optimizacion",
            "algoritmo": nombre,
            "dataset_version": VERSION,
            "uso": "experimental_no_produccion",
            "ia_responsable": "supervision_humana_requerida"
        })

        mlflow.log_params({
            "algoritmo": nombre,
            "cv_folds": len(pliegues),
            "criterio_optimizacion": "f1_macro",
            "semilla": 42,
            "registros_train": len(X_train),
            "caracteristicas": X_train.shape[1],
            "clases": len(clases_finales)
        })

        for parametro, valor in busqueda.best_params_.items():
            mlflow.log_param(parametro, valor)

        mlflow.log_metrics({
            "mejor_f1_macro_cv": float(
                busqueda.best_score_
            ),
            "tiempo_total_grid_seg": float(duracion),
            "numero_combinaciones": len(
                tabla_cv
            )
        })

        # Guardar evidencia de todas las combinaciones.
        mlflow.log_artifact(str(ruta_tabla))

        # Registrar el mejor modelo.
        # cloudpickle se utiliza solamente con modelos
        # generados dentro de esta sesión confiable.
        mlflow.sklearn.log_model(
            sk_model=busqueda.best_estimator_,
            name="mejor_modelo",
            serialization_format="cloudpickle"
        )

    resultados_grid.append({
        "modelo": nombre,
        "mejor_f1_macro_cv": busqueda.best_score_,
        "mejores_parametros": str(
            busqueda.best_params_
        ),
        "tiempo_grid_seg": duracion
    })

    print("\nMejores parámetros:")
    print(busqueda.best_params_)

    print("Mejor F1 macro CV:",
          round(busqueda.best_score_, 4))

    print("Tiempo total:",
          round(duracion, 2), "segundos")

    print("MLflow: registro completado")

# Comparación final de optimización.
df_resultados_grid = pd.DataFrame(resultados_grid)

print("\n=== COMPARACIÓN GRIDSEARCHCV ===")
print(df_resultados_grid.to_string(index=False))

ruta_final = EVID / "comparacion_gridsearch.csv"

df_resultados_grid.to_csv(
    ruta_final,
    index=False,
    encoding="utf-8-sig"
)

print("\nEvidencia:", ruta_final)
print("Modelos optimizados disponibles en memoria.")

print("\nIA RESPONSABLE:")
print("Conjunto de prueba reservado.")
print("Hiperparámetros registrados en MLflow.")
print("Resultados experimentales, no operativos.")


In [ ]:

# ============================================================
# CELDA 28 - Evaluación final de los modelos optimizados
# Actividad 5 | Control de acceso vehicular
#
# OBJETIVO:
# Evaluar LinearSVC y Random Forest en prueba independiente.
# Generar métricas, matrices de confusión y evidencias MLflow.
#
# IA RESPONSABLE:
# Documentar errores por clase y limitaciones.
# No utilizar estos resultados para reajustar hiperparámetros.
# ============================================================

import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

# 1. Validaciones previas.
assert set(groups_train).isdisjoint(set(groups_test))
assert len(X_test) == 2566
assert set(y_test).issubset(set(clases_finales))

etiquetas_ordenadas = sorted(clases_finales)
resultados_finales = []
predicciones_finales = {}

# 2. Evaluar ambos modelos, sin reentrenar.
for nombre, modelo in modelos_optimizados.items():

    print(f"\n=== EVALUACIÓN FINAL: {nombre} ===")

    inicio = time.perf_counter()
    predicciones = modelo.predict(X_test)
    duracion = time.perf_counter() - inicio

    predicciones_finales[nombre] = predicciones

    metricas = {
        "accuracy": accuracy_score(y_test, predicciones),
        "precision_macro": precision_score(
            y_test, predicciones,
            average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            y_test, predicciones,
            average="macro", zero_division=0
        ),
        "f1_macro": f1_score(
            y_test, predicciones,
            average="macro", zero_division=0
        ),
        "tiempo_inferencia_seg": duracion,
        "latencia_media_ms": (
            duracion / len(X_test) * 1000
        )
    }

    resultados_finales.append({
        "modelo": nombre,
        **metricas
    })

    # 3. Informe por clase.
    informe = classification_report(
        y_test, predicciones,
        labels=etiquetas_ordenadas,
        output_dict=True,
        zero_division=0
    )

    ruta_informe = (
        EVID / f"reporte_clases_{nombre.lower()}.csv"
    )

    pd.DataFrame(informe).T.to_csv(
        ruta_informe,
        encoding="utf-8-sig"
    )

    # 4. Matriz de confusión.
    matriz = confusion_matrix(
        y_test, predicciones,
        labels=etiquetas_ordenadas
    )

    fig, ax = plt.subplots(figsize=(13, 12))

    visualizador = ConfusionMatrixDisplay(
        confusion_matrix=matriz,
        display_labels=etiquetas_ordenadas
    )

    visualizador.plot(
        ax=ax,
        cmap="Blues",
        xticks_rotation=90,
        values_format="d",
        colorbar=False
    )

    ax.set_title(f"Matriz de confusión - {nombre}")
    plt.tight_layout()

    ruta_figura = (
        EVID / f"matriz_confusion_{nombre.lower()}.png"
    )

    fig.savefig(ruta_figura, dpi=160)
    plt.show()
    plt.close(fig)

    # 5. Registrar evaluación en MLflow.
    with mlflow.start_run(
        run_name=f"{nombre}_Evaluacion_Final"
    ):
        mlflow.set_tags({
            "actividad": "5",
            "etapa": "evaluacion_final",
            "dataset_version": VERSION,
            "modelo": nombre,
            "uso": "experimental_no_produccion",
            "ia_responsable": "evaluacion_de_errores_por_clase"
        })

        mlflow.log_metrics({
            f"test_{k}": float(v)
            for k, v in metricas.items()
        })

        mlflow.log_artifact(str(ruta_informe))
        mlflow.log_artifact(str(ruta_figura))

    print("Accuracy:", round(metricas["accuracy"], 4))
    print("F1 macro:", round(metricas["f1_macro"], 4))
    print("Latencia media (ms):",
          round(metricas["latencia_media_ms"], 3))
    print("Evidencias registradas en MLflow.")

# 6. Comparación general.
df_final = pd.DataFrame(resultados_finales)

print("\n=== RESULTADOS FINALES ===")
print(df_final.to_string(index=False))

ruta_final_test = EVID / "comparacion_final_test.csv"

df_final.to_csv(
    ruta_final_test,
    index=False,
    encoding="utf-8-sig"
)

print("\nEvidencia:", ruta_final_test)
print("Evaluación final completada.")

print("\nIA RESPONSABLE:")
print("Resultados limitados a clasificación de caracteres.")
print("No representan exactitud de matrícula completa.")
print("No autorizan la apertura automática de accesos.")


In [ ]:

# ============================================================
# CELDA 29 - Comparación estadística de modelos
# Actividad 5 | Control de acceso vehicular
#
# OBJETIVO:
# Comparar errores emparejados de LinearSVC y Random Forest.
#
# IA RESPONSABLE:
# Evitar afirmaciones sobre superioridad estadística
# sin evidencia y reconocer dependencia entre caracteres.
# ============================================================

import numpy as np
import pandas as pd
import json
import mlflow
from scipy.stats import binomtest
from sklearn.metrics import f1_score

# 1. Recuperar predicciones finales.
pred_svc = predicciones_finales["LinearSVC"]
pred_rf = predicciones_finales["RandomForest"]

assert len(pred_svc) == len(pred_rf) == len(y_test)

# 2. Identificar aciertos y errores por observación.
svc_correcto = pred_svc == y_test
rf_correcto = pred_rf == y_test

ambos_correctos = int(np.sum(svc_correcto & rf_correcto))
solo_svc = int(np.sum(svc_correcto & ~rf_correcto))
solo_rf = int(np.sum(~svc_correcto & rf_correcto))
ambos_incorrectos = int(np.sum(~svc_correcto & ~rf_correcto))

# 3. Prueba exacta de McNemar.
discordantes = solo_svc + solo_rf

if discordantes > 0:
    p_valor = binomtest(
        min(solo_svc, solo_rf),
        n=discordantes,
        p=0.5,
        alternative="two-sided"
    ).pvalue
else:
    p_valor = 1.0

# 4. Comparar métricas observadas.
f1_svc = f1_score(y_test, pred_svc, average="macro",
                  zero_division=0)
f1_rf = f1_score(y_test, pred_rf, average="macro",
                 zero_division=0)

accuracy_svc = float(np.mean(svc_correcto))
accuracy_rf = float(np.mean(rf_correcto))

print("=== COMPARACIÓN DE ACIERTOS ===")
print("Ambos correctos:", ambos_correctos)
print("Solo LinearSVC correcto:", solo_svc)
print("Solo Random Forest correcto:", solo_rf)
print("Ambos incorrectos:", ambos_incorrectos)

print("\n=== PRUEBA EXACTA DE MCNEMAR ===")
print("Casos discordantes:", discordantes)
print("Valor p:", round(float(p_valor), 6))

print("\n=== DIFERENCIAS OBSERVADAS ===")
print("Diferencia accuracy (puntos porcentuales):",
      round((accuracy_svc - accuracy_rf) * 100, 3))
print("Diferencia F1 macro (puntos porcentuales):",
      round((f1_svc - f1_rf) * 100, 3))

# 5. Guardar resumen reproducible.
resultado_estadistico = {
    "dataset_version": VERSION,
    "muestras_test": int(len(y_test)),
    "ambos_correctos": ambos_correctos,
    "solo_linearsvc_correcto": solo_svc,
    "solo_randomforest_correcto": solo_rf,
    "ambos_incorrectos": ambos_incorrectos,
    "p_valor_mcnemar_exacto": float(p_valor),
    "diferencia_accuracy_pp": float(
        (accuracy_svc - accuracy_rf) * 100
    ),
    "diferencia_f1_macro_pp": float(
        (f1_svc - f1_rf) * 100
    ),
    "limitacion": (
        "La prueba de McNemar supone observaciones "
        "independientes. Varios caracteres pueden "
        "pertenecer a la misma matricula; por ello, "
        "el valor p es exploratorio y no constituye "
        "inferencia definitiva a nivel de vehiculo."
    )
}

ruta_estadistica = EVID / "comparacion_estadistica.json"
ruta_estadistica.write_text(
    json.dumps(resultado_estadistico,
               indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# 6. Registrar estadística y advertencia en MLflow.
with mlflow.start_run(
    run_name="Comparacion_Estadistica_Final"
):
    mlflow.set_tags({
        "etapa": "analisis_comparativo",
        "dataset_version": VERSION,
        "ia_responsable": "limitaciones_estadisticas_documentadas",
        "resultado": "exploratorio"
    })
    mlflow.log_metrics({
        "p_valor_mcnemar_exacto": float(p_valor),
        "diferencia_accuracy_pp":
            resultado_estadistico["diferencia_accuracy_pp"],
        "diferencia_f1_macro_pp":
            resultado_estadistico["diferencia_f1_macro_pp"]
    })
    mlflow.log_artifact(str(ruta_estadistica))

print("\nEvidencia generada:", ruta_estadistica)
print("Registro estadístico completado en MLflow.")
print("\nIA RESPONSABLE:")
print("La dependencia entre caracteres limita la inferencia.")
print("No se modificaron modelos ni predicciones.")


In [ ]:

# ============================================================
# CELDA 30 - Respaldo de evidencias y experimentos
# Actividad 5 | Control de acceso vehicular
#
# IA RESPONSABLE:
# Preservar resultados, evitar pérdida de trazabilidad
# y no incluir imágenes originales ni datos personales.
# ============================================================

import os
import sqlite3
import zipfile
from pathlib import Path
from urllib.parse import urlparse, unquote
from datetime import datetime, timezone
import mlflow

BACKUP_DIR = RAIZ / "respaldos"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

FECHA = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RESPALDO = BACKUP_DIR / f"Actividad5_Respaldo_{FECHA}.zip"

# Copia consistente de SQLite mientras la sesión está activa.
DB_SNAPSHOT = BACKUP_DIR / "mlflow_snapshot.db"

with sqlite3.connect(str(DB_PATH)) as origen:
    with sqlite3.connect(str(DB_SNAPSHOT)) as destino:
        origen.backup(destino)

# Identificar artefactos locales registrados en MLflow.
cliente = mlflow.tracking.MlflowClient()
exp = mlflow.get_experiment_by_name(NOMBRE_EXPERIMENTO)

rutas_artefactos = set()

def ruta_local(uri):
    parsed = urlparse(uri)
    if parsed.scheme == "file":
        return Path(unquote(parsed.path))
    if parsed.scheme == "":
        return Path(uri)
    return None

runs = mlflow.search_runs(
    experiment_ids=[exp.experiment_id]
)

for run_id in runs["run_id"]:
    info = cliente.get_run(run_id)
    ruta = ruta_local(info.info.artifact_uri)
    if ruta is not None and ruta.exists():
        rutas_artefactos.add(ruta)

ruta_experimento = ruta_local(exp.artifact_location)
if ruta_experimento is not None and ruta_experimento.exists():
    rutas_artefactos.add(ruta_experimento)

incluidos = set()

with zipfile.ZipFile(
    RESPALDO, "w", compression=zipfile.ZIP_DEFLATED
) as z:

    # Reportes y evidencias generadas.
    for archivo in EVID.rglob("*"):
        if archivo.is_file():
            z.write(archivo, "evidencias/" + str(
                archivo.relative_to(EVID)
            ))
            incluidos.add(str(archivo.resolve()))

    # Base de datos SQLite.
    z.write(DB_SNAPSHOT, "mlflow/mlflow.db")

    # Artefactos locales.
    for carpeta in rutas_artefactos:
        archivos = (
            carpeta.rglob("*") if carpeta.is_dir() else [carpeta]
        )
        for archivo in archivos:
            if not archivo.is_file():
                continue
            if str(archivo.resolve()) in incluidos:
                continue
            z.write(
                archivo,
                "artefactos_locales/" +
                str(archivo.resolve()).lstrip("/")
            )
            incluidos.add(str(archivo.resolve()))

print("=== RESPALDO GENERADO ===")
print("Archivo:", RESPALDO)
print("Tamaño MB:", round(RESPALDO.stat().st_size / 1e6, 2))
print("Ejecuciones MLflow:", len(runs))
print("Rutas locales de artefactos:", len(rutas_artefactos))

with zipfile.ZipFile(RESPALDO) as z:
    archivos_zip = z.namelist()
    print("Archivos respaldados:", len(archivos_zip))
    print("Integridad ZIP:", z.testzip() or "CORRECTA")

print("\nIMPORTANTE:")
print("Respaldo creado en almacenamiento temporal.")
print("Hay que copiarlo a Google Drive antes de cerrar Colab.")
print("La restauración de artefactos requiere conservar sus rutas.")
print("El notebook y los datos .npz no están incluidos.")


In [ ]:

# ============================================================
# CELDA 31 - Respaldo persistente en Google Drive
# Actividad 5 | Control de acceso vehicular
#
# OBJETIVO:
# Conservar los experimentos y sus evidencias fuera
# del almacenamiento temporal de Google Colab.
#
# IA RESPONSABLE:
# Protección de la trazabilidad y acceso restringido.
# ============================================================

from google.colab import drive
from pathlib import Path
import shutil
import hashlib

# 1. Conectar Google Drive.
drive.mount("/content/drive")

# 2. Crear carpeta de respaldo.
DESTINO = Path(
    "/content/drive/MyDrive/"
    "Actividad5_ControlAccesoVehicular/respaldos"
)

DESTINO.mkdir(parents=True, exist_ok=True)

# 3. Copiar ZIP existente.
archivo_destino = DESTINO / RESPALDO.name

shutil.copy2(RESPALDO, archivo_destino)

# 4. Verificar integridad.
hash_origen = sha256_archivo(RESPALDO)
hash_destino = sha256_archivo(archivo_destino)

assert hash_origen == hash_destino, (
    "ERROR: el respaldo no coincide con el original."
)

print("=== RESPALDO EN GOOGLE DRIVE ===")
print("Archivo:", archivo_destino)
print("Tamaño MB:", round(
    archivo_destino.stat().st_size / 1e6, 2
))
print("Verificación SHA-256: CORRECTA")

print("\nRESPALDO PERSISTENTE COMPLETADO")
print("IA RESPONSABLE: conservar carpeta con acceso restringido.")


### Condiciones para avanzar al Notebook 02
1. ZIP íntegro y trazable a Zenodo.
2. Asociación fiable JSON ↔ imagen y lectura válida de coordenadas.
3. Auditoría de duplicados y posibles fugas entre particiones (vehículo y/o matrícula).
4. Licencia y tratamiento de matrículas documentados.
5. Revisión humana de una muestra de recortes.

**No ejecutar GridSearchCV ni reportar métricas todavía.** Los resultados de este notebook son de auditoría preliminar, no de desempeño de modelos.